# Microbiome growth landscape analysis

We’ll analyse an empirical growth landscape to explore how microbial identity and nutrient choice shape an experimental response. A screen of gut bacterial strains grown on medicinal polysaccharides will serve as our example.

## 1. Importing necessary dependencies

Install GraphFLA with pip, then import the dependencies below.

In [ ]:
!pip install graphfla

In [1]:
import graphfla
from graphfla.landscape import Landscape
import pandas as pd

## 2. Loading the experimental dataset

For our example, we use the BacPUS screen from [Qu et al. (2025)](https://doi.org/10.1038/s41467-025-55845-7). The dataset pairs 28 human gut bacterial isolates with 20 polysaccharide preparations. Each isolate was grown separately with a polysaccharide as the carbon source.

We’ll use mean optical density at 600 nm after 48 hours (**OD600**) as the growth score. The curated table averages available replicate measurements. Small negative values reflect background-corrected readings near zero and are retained.

| Column | Meaning |
|---|---|
| `strain_id` | Identifier of the bacterial isolate. |
| `strain` | Species name associated with that isolate. |
| `polysaccharide` | Identifier of the polysaccharide preparation. |
| `polysaccharide_name` | Descriptive name of the preparation. |
| `OD600_48h` | Mean optical density after 48 hours; larger indicates more growth in this assay. |

Let’s load the strain–substrate measurements.

In [2]:
data_url = (
    "https://raw.githubusercontent.com/COLA-Laboratory/GraphFLA/"
    "main/data/Microbiome/MedPoly/growth.csv"
)
df = pd.read_csv(data_url)[[
    "strain_id", "strain", "polysaccharide", "polysaccharide_name", "OD600_48h",
]]
df.head()

,strain_id,strain,polysaccharide,polysaccharide_name,OD600_48h
0,DA103,Bacteroides_massiliensis,APs,Astragalus polysaccharides,0.030000
1,DA103,Bacteroides_massiliensis,AdPs,Adenophora polysaccharides,-0.001833
2,DA103,Bacteroides_massiliensis,AnPs,Angelica polysaccharides,0.010833
3,DA103,Bacteroides_massiliensis,CPPs,Codonopsis pilosula polysaccharides,-0.003333
4,DA103,Bacteroides_massiliensis,DPs_1,Dendrobe polysaccharides 1,0.084500


## 3. Preparing the inputs

To construct a landscape, GraphFLA only needs two inputs—much like the features `X` and targets `y` used to train a machine-learning model:

- `X`: the experimental configurations, as a DataFrame or a two-dimensional NumPy array.
- `f`: one measured outcome per configuration, as a Series, list or one-dimensional array.

The rows of `X` and `f` must refer to the same configurations in the same order.


We select `strain_id` and `polysaccharide` as `X`, and the 48-hour OD600 measurement as `f`. Strain IDs keep distinct isolates of the same species separate; using the species name alone would merge some experimental configurations.

In [3]:
X = df[['strain_id', 'polysaccharide']]
f = df['OD600_48h']
X.head()

,strain_id,polysaccharide
0,DA103,APs
1,DA103,AdPs
2,DA103,AnPs
3,DA103,CPPs
4,DA103,DPs_1


## 4. Constructing the landscape

With the inputs ready, we can use `Landscape` with two categorical variables. Neighbouring configurations replace the strain or the polysaccharide while keeping the other choice fixed. These are comparisons between assay configurations, not evolutionary transitions between species.

We set `maximize=True` because greater growth is our objective.

In [4]:
landscape = Landscape(maximize=True)
landscape.build_from_data(
    X, f,
    data_types={"strain_id": "categorical", "polysaccharide": "categorical"},
)

Building Landscape from data...


 - Validating data types dictionary...


   - Data types dictionary validation successful.


 - Handling missing values and duplicates...


 - Checking for missing values...


...

   - Found 3 local optima (3 member nodes: 0 plateau-LOs + 3 single-point LOs).


 - Determining global optimum...


   - Global optimum found at index 308 with fitness 0.6927.


Landscape built successfully.



Landscape(kind='default', maximize=True)

## 5. Inspecting the landscape

Let’s first look at what we’ve built. Printing the landscape shows its variables, configurations, improving edges and local optima:

In [5]:
print(landscape)

Landscape(kind='default'): 2 variables, 560 configurations, 12832 edges, 3 local optima


For a closer look at individual strain–substrate combinations, we can call `get_data()`. The selected outcome appears as `fitness`; `out_degree` counts directly improving moves, `in_degree` counts incoming moves from lower-scoring neighbours, and `is_lo` indicates local-optimum membership.

In [6]:
landscape.get_data()

,strain_id,polysaccharide,fitness,plateau_id,plateau_size,in_degree,out_degree,is_lo
0,DA103,APs,0.030000,-1,1,14,32,False
1,DA103,AdPs,-0.001833,-1,1,6,40,False
2,DA103,AnPs,0.010833,-1,1,11,35,False
3,DA103,CPPs,-0.003333,-1,1,3,43,False
4,DA103,DPs_1,0.084500,-1,1,35,11,False
...,...,...,...,...,...,...,...,...
555,DA904,PSPs,0.013000,-1,1,12,34,False
556,DA904,RAPs,0.024833,-1,1,13,33,False
557,DA904,RNPs,0.049333,-1,1,16,30,False
558,DA904,RPPs,0.073833,-1,1,32,14,False


We can also summarise the growth measurements with pandas:

In [7]:
landscape.get_data()["fitness"].describe()

count    560.000000
mean       0.132891
std        0.146531
min       -0.011667
25%        0.020167
50%        0.074667
75%        0.191250
max        0.692667
Name: fitness, dtype: float64

To inspect the strain–substrate combination with the highest growth score, we can use the global-optimum node index:

In [8]:
landscape[landscape.go_index]

{'strain_id': 'DA320',
 'polysaccharide': 'GPs',
 'fitness': np.float64(0.6926666666666668),
 'plateau_id': -1,
 'plateau_size': 1,
 'in_degree': 46,
 'out_degree': 0,
 'is_lo': True}

## 6. Analysing the landscape

Next, we’ll count local optima, measure ruggedness, examine interactions and assess the trend toward the best observed configuration.

In [9]:
from graphfla import analysis

### 6.1 Number of local optima

We can start with `landscape.n_lo`. A local optimum has no improving route out of its immediate neighbourhood, including neutral moves within a plateau. Each connected neutral optimum plateau counts once. Multiple local optima mean that successive improvements can lead to different endpoints.

In [10]:
print(f"Number of local optima: {landscape.n_lo}")

Number of local optima: 3


### 6.2 Roughness-to-slope ratio

For a view of ruggedness, we can calculate `r_s_ratio`. It compares deviations from an additive model with the average magnitude of its coefficients. Larger values indicate more residual roughness relative to the additive trend; a nearly zero fitted slope can give an infinite ratio.


Here, the additive model assigns independent contributions to strain identity and substrate identity. Residual variation measures what this simple model leaves unexplained.

In [11]:
roughness_to_slope = analysis.r_s_ratio(landscape)
print(f"Roughness-to-slope ratio: {roughness_to_slope:.4f}")

Roughness-to-slope ratio: 1.0875


### 6.3 Strain–substrate interactions

A substrate that supports one strain may support another poorly. We can summarise this context dependence with `classify_epistasis()`. The interaction is between strain identity and substrate choice; these single-isolate assays do not measure interactions between bacteria in a mixed community.

We can explore these patterns with `classify_epistasis()`, which classifies four-configuration graph motifs:

| Category | Interpretation |
|---|---|
| `magnitude` | Changes retain their direction of effect across contexts. |
| `sign` | One change reverses its direction of effect across contexts. |
| `reciprocal_sign` | Both changes reverse their direction of effect across contexts. |

The output gives their proportions among the detected motifs. The `positive` and `negative` fields form a separate partition.

In [12]:
interactions = analysis.classify_epistasis(landscape, seed=42)
interactions

EpistasisClassification(magnitude=0.3959890559332073, sign=0.3928722534059177, reciprocal_sign=0.21113869066087496, positive=0.8948184920881167, negative=0.1051815079118833)

### 6.4 Fitness-distance correlation

Finally, we can use `fdc` to check whether higher-scoring configurations tend to lie closer to the best observed configuration—a simple view of navigability.


Distance counts how many choices—strain identity and polysaccharide identity—differ from the best observed combination. It does not encode taxonomic or chemical similarity.

We use Spearman correlation to compare ranks. A negative value means higher scores tend to occur nearer the optimum; a value near zero indicates little monotonic association. This trend does not guarantee an improving path from every configuration.

In [13]:
fitness_distance_r = analysis.fdc(landscape, method="spearman")
print(f"Fitness-distance correlation: {fitness_distance_r:.4f}")

 - Calculating distances to global optimum using mixed_distance...


   - Distances to GO calculated and added as node attribute 'dist_go'.


Fitness-distance correlation: -0.2184


## 7. Running several analyses together

Once we’ve tried the metrics individually, we can collect them with `analysis.profile()`. Here, we combine the ruggedness measure and fitness-distance correlation in a pandas Series. Setting `on_error="raise"` displays errors immediately.

In [14]:
analysis.profile(
    landscape,
    include=["r_s_ratio", "fdc"],
    on_error="raise",
    progress=False,
)

r_s_ratio    1.087480
fdc         -0.218363
dtype: float64

## 8. Analysis reference

For further exploration, we can choose from the **30 analysis functions** below. Each takes a built `landscape` as its first argument. In these functions, *fitness* refers to the outcome selected as `f`.

Walsh–Hadamard and higher-order regression are included for reference. Their cost grows rapidly with the number of variables and levels, so we leave them out of the worked examples.

| Group | Function | Result |
| --- | --- | --- |
| Fitness | `fitness_distribution` | Dictionary of distribution summaries, including skewness, kurtosis, CV and Cauchy location. |
| Fitness | `fitness_effect_distribution` | List of signed effects for one mutation across matched backgrounds. |
| Ruggedness | `local_optima_ratio` | Local-optimum count divided by configuration count. |
| Ruggedness | `r_s_ratio` | Additive-model residual roughness divided by mean absolute coefficient. |
| Ruggedness | `autocorrelation` | Fitness autocorrelation along sampled walks. |
| Ruggedness | `gradient_intensity` | Mean absolute change on improving edges divided by mean fitness. |
| Correlation | `fdc` | Fitness-distance correlation: Spearman or Pearson. |
| Correlation | `neighbor_fitness_correlation` | Correlation between fitness and mean neighbour fitness: Pearson, Spearman or Kendall. |
| Correlation | `basin_fitness_correlation` | Correlation between peak fitness and greedy basin size: Spearman or Pearson. |
| Correlation | `fitness_flattening_index` | Trend in successive fitness gains along greedy walks: Spearman or Pearson. |
| Navigability | `global_optima_accessibility` | Fraction of configurations with an improving route to the global optimum. |
| Navigability | `local_optima_accessibility` | DataFrame of accessibility values for specified local optima. |
| Navigability | `mean_distance_to_global_optimum` | Mean sequence/configuration distance to the global optimum. |
| Navigability | `mean_distance_to_local_optima` | DataFrame of mean sequence/configuration distances to specified local optima. |
| Navigability | `mean_path_length_to_global_optimum` | Mean shortest graph-path length, excluding unreachable starts. |
| Navigability | `mean_path_length_to_local_optima` | DataFrame of shortest-path summaries, excluding unreachable starts. `lo=None` selects the global optimum. |
| Robustness | `neutrality` | Fraction of neighbour pairs within the fitness-difference threshold. |
| Robustness | `evolvability_enhancing_fraction` | Fraction of all directed neighbour pairs classified as EE using non-focal neighbourhoods and BH-corrected tests. |
| Robustness | `single_mutation_effects` | DataFrame of mutation-pair effects and one-sided tests at one position; positive or negative tests. |
| Robustness | `all_mutation_effects` | DataFrame of mutation-pair effects and one-sided tests at all positions. |
| Epistasis | `idiosyncratic_index` | Normalized spread of one mutation’s effects across backgrounds. |
| Epistasis | `global_idiosyncratic_index` | Mean idiosyncratic index across eligible mutations. |
| Epistasis | `gamma` | Correlation of mutation effects across matched two-site backgrounds. |
| Epistasis | `gamma_star` | Consistency of mutation-effect signs across backgrounds. |
| Epistasis | `diminishing_returns_index` | Association of background fitness with mean beneficial gain: Pearson, Spearman or regression slope. |
| Epistasis | `increasing_costs_index` | Association of background fitness with mean deleterious cost: Pearson, Spearman or regression slope. |
| Epistasis | `walsh_hadamard` | Coefficients and an order summary with fit gains and the fitted-model variance spectrum. |
| Epistasis | `classify_epistasis` | Object with magnitude, sign, reciprocal-sign, positive and negative epistasis proportions. |
| Epistasis | `extradimensional_bypass` | Object with bypass proportion, mean length and motif counts. |

### Data source

Qu Z., Liu H., Yang J. et al. (2025). [*Selective utilization of medicinal polysaccharides by human gut Bacteroides and Parabacteroides species*. Nature Communications 16, 638](https://doi.org/10.1038/s41467-025-55845-7).

The repository CSV summarizes the authors’ growth measurements at 48 hours, retaining isolate identifiers and averaging available replicates.